# Chapter 6 – Statistical Machine Learning

## 📖 Summary
Chapter 6 covers more advanced predictive methods from the machine learning tradition, with a statistical perspective on how and why they work.

### Key Topics Covered:
1. **K-Nearest Neighbors (KNN)** – instance-based learning
2. **Tree Models** – decision trees, splitting criteria
3. **Bagging & Random Forest** – ensemble averaging
4. **Boosting** – AdaBoost, Gradient Boosting
5. **Regularization & Hyperparameter Tuning**
6. **Feature Importance**
7. **Bias-Variance Tradeoff**


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier, BaggingClassifier)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## 2. Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train, y_train)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, dt.predict(X_test)):.4f}")
print(f"Decision Tree AUC:      {roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]):.4f}")

plt.figure(figsize=(20, 6))
plot_tree(dt, feature_names=data.feature_names, class_names=data.target_names,
          filled=True, rounded=True, max_depth=2, fontsize=9)
plt.title('Decision Tree (depth=2 shown)')
plt.tight_layout()
plt.savefig('ch6_tree.png', dpi=100, bbox_inches='tight')
plt.show()


### 📚 Theory: Decision Trees
A decision tree recursively partitions the feature space by selecting the **best split** at each node.

**Splitting criteria:**
- **Gini Impurity**: $G = 1 - \sum_k p_k^2$ — probability of misclassifying a random sample
- **Entropy**: $H = -\sum_k p_k \log_2(p_k)$ — information theoretic measure

Trees grow until a stopping criterion (max depth, min samples) is met. Advantages: interpretable, handles mixed types. Drawback: high variance (overfitting).


## 3. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42)
rf.fit(X_train, y_train)
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}")
print(f"Random Forest AUC:      {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}")

# Feature Importance
feat_imp = pd.Series(rf.feature_importances_, index=data.feature_names).sort_values(ascending=False)[:10]

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.xlabel('Feature Importance (Mean Decrease Impurity)')
plt.title('Top 10 Features – Random Forest')
plt.tight_layout()
plt.savefig('ch6_rf_importance.png', dpi=100, bbox_inches='tight')
plt.show()


### 📚 Theory: Random Forest
**Bagging** (Bootstrap AGGregation): train $B$ trees on bootstrap samples and **average** predictions.

**Random Forest** improves bagging by also randomly selecting $m$ features at each split (typically $m = \sqrt{p}$ for classification):

- Reduces **correlation** between trees → lower ensemble variance
- More features considered → more diverse trees → better generalization
- Feature importance = average reduction in impurity across all trees and all splits

$$\hat{f}_{RF}(x) = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$


## 4. Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                max_depth=4, random_state=42)
gb.fit(X_train, y_train)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, gb.predict(X_test)):.4f}")
print(f"Gradient Boosting AUC:      {roc_auc_score(y_test, gb.predict_proba(X_test)[:,1]):.4f}")

# Learning curve: AUC vs number of trees
train_aucs, test_aucs = [], []
for n in range(1, 101):
    gb_n = GradientBoostingClassifier(n_estimators=n, learning_rate=0.1, max_depth=4, random_state=42)
    gb_n.fit(X_train, y_train)
    train_aucs.append(roc_auc_score(y_train, gb_n.predict_proba(X_train)[:,1]))
    test_aucs.append(roc_auc_score(y_test, gb_n.predict_proba(X_test)[:,1]))

plt.figure(figsize=(9, 4))
plt.plot(train_aucs, label='Train AUC')
plt.plot(test_aucs, label='Test AUC')
plt.xlabel('Number of Trees')
plt.ylabel('AUC')
plt.title('Gradient Boosting – Learning Curve')
plt.legend()
plt.tight_layout()
plt.savefig('ch6_gb_curve.png', dpi=100, bbox_inches='tight')
plt.show()


### 📚 Theory: Gradient Boosting
**Boosting** builds trees **sequentially**, where each tree corrects errors of the previous ensemble.

Gradient Boosting fits a tree to the **negative gradient** of the loss function:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where $h_m$ is the $m$-th tree and $\eta$ is the **learning rate**.

- Smaller $\eta$ + more trees → better generalization but slower
- Key hyperparameters: `n_estimators`, `learning_rate`, `max_depth`, `subsample`


## 5. Bias-Variance Tradeoff

In [ ]:
# Show bias-variance tradeoff via tree depth
depths = range(1, 20)
train_scores, test_scores = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(depths, train_scores, 'o-', label='Train Accuracy', color='steelblue')
plt.plot(depths, test_scores,  'o-', label='Test Accuracy',  color='tomato')
plt.axvline(4, color='green', linestyle='--', label='Sweet spot')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Bias-Variance Tradeoff: Tree Depth vs Accuracy')
plt.legend()
plt.tight_layout()
plt.savefig('ch6_bias_variance.png', dpi=100, bbox_inches='tight')
plt.show()


### 📚 Theory: Bias-Variance Tradeoff

$$\text{Expected Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

- **High Bias** (underfitting): model too simple, misses patterns
- **High Variance** (overfitting): model too complex, memorizes training noise
- **Goal**: find the sweet spot that minimizes total test error

Ensembles (RF, GB) reduce variance without significantly increasing bias.


## ✅ Chapter 6 Summary

| Model | Bias | Variance | Interpretability |
|---|---|---|---|
| Decision Tree (deep) | Low | High | High |
| Decision Tree (shallow) | High | Low | Very High |
| Random Forest | Low | Low | Medium |
| Gradient Boosting | Very Low | Low | Low |

**Key takeaway**: Random Forests are robust defaults. Gradient Boosting often achieves highest accuracy but requires careful tuning. Always use cross-validation for model selection.
